# Prepare and verify training and holdout datasets

Prepare condition-classification training and holdout JSONL from the bundled positive and negative records. The notebook calls the main dataset preparation module through `mof_dataset_preparation_demo.py`. No API key or training job is needed.

Install from the repository root with `python -m pip install -e ".[curation,datasets,notebook]"`.

## Inputs and setup

Run the cells in order. The following paths are relative to `Demo/02_dataset_preparation/`; all required files are included.

| Input | Location | Contents |
| --- | --- | --- |
| Processed positive records, CSV | `input/processed_positive.csv` | The expected processed positive output from the data curation demonstration. |
| Processed negative records, CSV | `input/processed_negative.csv` | Enumerated and curated negative-condition records. |
| Publication years, CSV | `input/publication_years.csv` | `DOI` and `Publication Year` columns. |
| Holdout conditions, JSON | `input/forced_questions.json` | Condition definitions selected by `config.json`. |
| Classification prompt, TXT | `../../prompts/training/reaction_prediction.txt` | The main workflow's classification prompt. |

To use the output of the data curation demonstration, pass `positive_csv=REPO / "Demo/01_data_curation/outputs/mof_extraction_6.csv"` to `run_demo` below. To test different processed tables, select their paths in `config.json` and set `CHECK_EXPECTED = False`; expected-output verification applies only to the supplied dataset. Keep the input column schemas and provide matching DOI/year metadata.

Outputs are saved together under `outputs/`: `mof_ft_train.jsonl`, `mof_ft_holdout.jsonl`, `mof_ft_split_assignments.csv`, and summary files. The split is computed locally; it does not start a training job.

Implementation: [demonstration runner](mof_dataset_preparation_demo.py) and [dataset preparation](../../src/mofinder/datasets/prepare.py). See the [source-to-code guide](../../docs/source_to_code.md) to find the original source and its corresponding Python functions.


In [1]:
from pathlib import Path
import runpy
import pandas as pd
from IPython.display import display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "mofinder").is_dir():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from within the MOFinder repository.")

import json
DEMO = REPO / "Demo" / "02_dataset_preparation"
CHECK_EXPECTED = True  # Set False for custom inputs or settings.


## 1. Inspect the inputs

In [2]:
positive = pd.read_csv(DEMO / "input" / "processed_positive.csv")
negative = pd.read_csv(DEMO / "input" / "processed_negative.csv")
pd.DataFrame({"label": ["P", "N"], "input_rows": [len(positive), len(negative)]})


,label,input_rows
0,P,146
1,N,175


## 2. Prepare the datasets

The seed-42 split groups records by metal precursor, linker set, and solvent set. The bundled positive input is the expected output of the data curation demonstration. The summary below shows the generated P and N counts for each split.


In [3]:
runner = runpy.run_path(str(DEMO / "mof_dataset_preparation_demo.py"))
run_demo = runner["run"]
summary = run_demo(check=CHECK_EXPECTED)
pd.DataFrame({name: summary["labels"][name] for name in ["train", "holdout"]})


Training: 252 records; holdout: 28 records.
Shared clusters: 0
Output: MOFinder\Demo\02_dataset_preparation\run_history\20260925T185722.490689Z_02a92ce2\outputs
Expected JSONL, labels, and split assignments: PASS


Run record: Demo/02_dataset_preparation/run_history/20260925T185722.490689Z_02a92ce2/run_record.json


,train,holdout
P,108,12
N,144,16


## 3. Verify against expected output

Rerun this cell at any time after generating the files. It reads the current files in `outputs/`, compares them with the bundled `expected/` files, and displays expected and actual row counts with **PASS** or **FAIL**. A matching row count alone is insufficient: the check also compares the file contents. A failed check stops here with an assertion error and includes the reason in the table.

These expected results apply to the supplied inputs and settings. For your own data, set `CHECK_EXPECTED = False` in the setup cell. This skips both the run-time check and the separate comparison below.


In [4]:
if CHECK_EXPECTED:
    verification = runner["verify_outputs"](DEMO / "outputs")
    checks = pd.DataFrame(verification["checks"])
    checks["result"] = checks["passed"].map({True: "PASS", False: "FAIL"})
    display(checks[["file", "expected_rows", "actual_rows", "result", "detail"]])
    assert verification["passed"], "Output verification failed; inspect the comparison above."
else:
    print("Expected-output verification skipped (CHECK_EXPECTED = False).")


,file,expected_rows,actual_rows,result,detail
0,mof_ft_train.jsonl,252.0,252.0,PASS,"All JSONL records match, including order and l..."
1,mof_ft_holdout.jsonl,28.0,28.0,PASS,"All JSONL records match, including order and l..."
2,mof_ft_class_map.json,NaN,NaN,PASS,All JSON values match the expected output.
3,demo_summary.json,NaN,NaN,PASS,All JSON values match the expected output.
4,mof_ft_split_assignments.csv,280.0,280.0,PASS,"All columns, rows, and values match the expect..."


## 4. Compare a few holdout and training examples

Show **two positive (P) and two negative (N) examples per split**, with holdout first. Change `N_PER_LABEL` below to display more or fewer examples. Within each split and label, examples are ordered by `source_row_id`.

The negative labels refer to reconstructed conditions; they do not establish that every condition was individually tested and failed.

The JSONL files are shuffled. The preview matches each assignment to its saved JSONL record by normalized reaction conditions and label, rather than assuming their row numbers agree. `jsonl_line` is the 1-based line number in that split's JSONL file.


In [5]:
from collections import defaultdict
from html import escape
from IPython.display import HTML, Markdown, display
from mofinder.datasets.prepare import forced_question_condition_key

N_PER_LABEL = 2  # Up to two P and two N examples from each split.
if type(N_PER_LABEL) is not int or N_PER_LABEL < 1:
    raise ValueError("N_PER_LABEL must be a positive integer.")

assignments = pd.read_csv(
    DEMO / "outputs" / "mof_ft_split_assignments.csv",
    dtype={"is_success": "boolean", "publication_year": "Int64"},
)
assignments["label"] = assignments["is_success"].map({True: "P", False: "N"})

# Index the actual saved records; keep chemical spelling from the user-message JSON.
records_by_key = defaultdict(list)
for split in ["holdout", "train"]:
    with (DEMO / "outputs" / f"mof_ft_{split}.jsonl").open(encoding="utf-8-sig") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            conditions = json.loads(next(m["content"] for m in record["messages"] if m["role"] == "user"))
            label = next(m["content"] for m in record["messages"] if m["role"] == "assistant").strip()
            key = (split, forced_question_condition_key(conditions), label)
            records_by_key[key].append((line_number, conditions, record))

preview_rows = []
selected_examples = []
for split in ["holdout", "train"]:
    for label in ["P", "N"]:
        selected = assignments.loc[
            assignments["split"].eq(split) & assignments["label"].eq(label)
        ].sort_values("source_row_id").head(N_PER_LABEL)
        for row in selected.itertuples(index=False):
            key = (split, forced_question_condition_key(json.loads(row.condition_key)), label)
            matches = records_by_key.get(key, [])
            if len(matches) != 1:
                raise ValueError(f"Expected one JSONL match for source row {row.source_row_id}; found {len(matches)}.")
            line_number, conditions, record = matches[0]
            metadata = {
                "split": split, "source_row_id": row.source_row_id,
                "doi_norm": row.doi_norm, "publication_year": row.publication_year,
                "is_success": row.is_success, "label": label, "jsonl_line": line_number,
            }
            preview_rows.append(metadata)
            selected_examples.append({**metadata, "conditions": conditions, "record": record})

preview = pd.DataFrame(preview_rows)
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(preview)

,split,source_row_id,doi_norm,publication_year,is_success,label,jsonl_line
0,holdout,2,10.1038/s41560-018-0261-6,2018,True,P,23
1,holdout,65,10.1039/b713705b,2008,True,P,9
2,holdout,221,10.1021/ic000209d,2000,False,N,11
3,holdout,225,10.1021/ic000209d,2000,False,N,25
4,train,0,10.1038/s41560-018-0261-6,2018,True,P,33
5,train,1,10.1038/s41560-018-0261-6,2018,True,P,84
6,train,146,10.1021/acs.cgd.6b01732,2017,False,N,157
7,train,147,10.1021/acsami.5c01476,2025,False,N,106


## 5. Inspect reaction parameters and the full JSON records

Each column below is one of the selected examples. The eight parameters come from its saved user message, with units in the field names. Expand a record to see the complete system/user/assistant JSON. The assistant P/N value is the dataset label, not a new model prediction.


In [6]:
for split in ["holdout", "train"]:
    examples = [item for item in selected_examples if item["split"] == split]
    display(Markdown(f"### {'Holdout' if split == 'holdout' else 'Training'} examples"))
    if not examples:
        print("No examples in this split.")
        continue

    parameters = pd.DataFrame({
        f"Source {item['source_row_id']} | {item['label']}": item["conditions"]
        for item in examples
    })
    parameters.index.name = "Reaction parameter"
    # Wrap long reagent names and display all values without pandas truncation.
    display(parameters.style.set_properties(**{
        "text-align": "left", "white-space": "normal", "overflow-wrap": "anywhere",
    }).format(str, na_rep="null"))

    for index, item in enumerate(examples):
        title = (f"Source {item['source_row_id']} | {item['label']} | {item['doi_norm']} | "
                 f"mof_ft_{split}.jsonl, line {item['jsonl_line']}")
        opened = " open" if index == 0 else ""
        display(HTML(
            f"<details{opened}><summary>{escape(title)}</summary>"
            '<pre style="white-space:pre-wrap;overflow-wrap:anywhere">'
            + escape(json.dumps(item["record"], ensure_ascii=False, indent=2))
            + "</pre></details>"
        ))

### Holdout examples

,Source 2 | P,Source 65 | P,Source 221 | N,Source 225 | N
Reaction parameter,,,,
metal_precursor,ZrOCl2·8H2O,Cu2(MeCOO)4·2H2O,Ce(NO3)3·6H2O,Eu(NO3)3·6H2O
organic_linker,"benzene-1,3,5-tricarboxylic acid","aH-PTMHC and 4,4'-bipyridine","cis,cis-1,3,5-cyclohexanetricarboxylic acid","cis,cis-1,3,5-cyclohexanetricarboxylic acid"
modulator,formic acid,null,null,null
solvent,dimethylformamide,water and ethanol,water,water
metal_concentration_mM,52.0,10.0,53.0,53.0
M_L_ratio,0.95,1.51,0.8,0.8
temperature_C,110.0,25.0,170.0,170.0
time_h,48.0,480.0,24.0,24.0


### Training examples

,Source 0 | P,Source 1 | P,Source 146 | N,Source 147 | N
Reaction parameter,,,,
metal_precursor,ZrCl4,ZrCl4,Cu(NO3)2·1H2O,FeCl3·6H2O
organic_linker,tetrakis(4-carboxyphenyl)porphyrin,tetrakis(4-carboxyphenyl)porphyrin,tetracarboxylic meta-substituted four-wall aryl-extended calix[4]pyrrole,2-aminoterephthalic acid
modulator,benzoic acid,benzoic acid,null,null
solvent,"N,N-diethylformamide",dimethylformamide,dimethylformamide,ethanol
metal_concentration_mM,40.0,64.0,5.0,203.0
M_L_ratio,5.09,10.18,0.5,1.42
temperature_C,120.0,120.0,100.0,25.0
time_h,48.0,24.0,24.0,12.0


The full split summary records input hashes, filtering counts, and preparation settings. The small demonstration datasets are separate from the full research training and holdout files.

## Saved run record

The executed output in this notebook preserves a readable example. The checked-in [recorded runs](recorded_runs/README.md) preserve run records and output snapshots for inspection on GitHub.

Each new run also saves its own timestamped folder under `run_history/`, including its output snapshot and run record. `outputs/` contains the latest generated files; earlier runs remain in `run_history/`. Local history is excluded from Git by default.


In [7]:
print("Saved run record:", summary["run_record"])
run_verification = summary.get("verification")
print("Verification:", "Not requested" if run_verification is None else ("PASS" if run_verification["passed"] else "FAIL"))


Saved run record: Demo/02_dataset_preparation/run_history/20260925T185722.490689Z_02a92ce2/run_record.json
Verification: PASS
